In [9]:
import dynamiqs as dq
import jax.numpy as jnp
from matplotlib import pyplot as plt

from jax import vmap, jit
from cmaes import SepCMA

from scipy.optimize import curve_fit
from scipy.optimize import least_squares

# dq.set_progress_meter(False)

In [ ]:
def measure_lifetime(initial_state, tfinal, g_2, eps_d):
    na = 15 # Hilbert space dimension
    nb = 5
    a = dq.tensor(dq.destroy(na), dq.eye(nb)) # annihilaiton operator
    b = dq.tensor(dq.eye(na), dq.destroy(nb))

    kappa_b = 10 # MHz
    #eps_d = 4
    #g_2 = 1 # MHz  
    kappa_a = 1 # MHz

    eps_2 = 2 * g_2 * eps_d / kappa_b
    kappa_2 = 4 * jnp.abs(g_2)**2/kappa_b
    alpha_estimate = jnp.sqrt(2/kappa_2 * (eps_2 - kappa_a/4))

    #print(f"Estimated cat size: {alpha_estimate:.2f}")

    H = jnp.conj(g_2) * a @ a @ b.dag() + g_2 * a.dag() @ a.dag() @ b - eps_d * b.dag() - jnp.conj(eps_d) * b

    loss_b = jnp.sqrt(kappa_b) * b
    loss_a = jnp.sqrt(kappa_a) * a

    tsave = jnp.linspace(0, tfinal, 3)

    g_state = dq.coherent(na, alpha_estimate)
    e_state = dq.coherent(na, -alpha_estimate)

    basis = {
        "+z": g_state,
        "-z": e_state,
        "+x": (g_state + e_state) / jnp.sqrt(2),
        "-x": (g_state - e_state) / jnp.sqrt(2),
        "+y": (g_state + 1j*e_state) / jnp.sqrt(2),
        "-y": (g_state - 1j*e_state) / jnp.sqrt(2),
    }

    sx = (1j * jnp.pi * a.dag() @ a).expm()
    
    # This construction of sigmaz will not work without a good estimate of alpha, which is hard to come by in experiment.
    sz = basis["+z"] @ basis["+z"].dag() - basis["-z"] @ basis["-z"].dag()
    sz = dq.tensor(sz, dq.eye(nb))

    psi0 = dq.tensor(basis[initial_state], dq.fock(nb,0)) # initial state

    res = dq.mesolve(
        H, 
        [loss_b, loss_a], 
        psi0, 
        tsave, 
        options=dq.Options(progress_meter=False),
        exp_ops=[sx, sz]
    )

    return res

In [35]:
# model: y = A * exp(-t/tau) + C
def model(p, t):
    A, tau, C = p
    return A * jnp.exp(-t/tau) + C

def residuals(p, x, y):
    return model(p, x) - y


def robust_exp_fit(x, y):
    # smart initialization
    A0 = y.max() - y.min()
    C0 = y.min()
    tau0 = (x.max() - x.min())
    p0 = [A0, tau0, C0]

    # robust fit (soft_l1 or huber are key)
    res = least_squares(
        residuals,
        p0,
        args=(x, y),
        bounds=([0, 0, -jnp.inf], [jnp.inf, jnp.inf, jnp.inf]),
        loss="soft_l1",   # try "huber" too
        f_scale=0.1       # tune based on noise level
    )

    A, tau, C = res.x
    y_fit = model(res.x, x)

    return {
        "popt": res.x,
        "y_fit": y_fit,
    }

In [36]:
def _lossfunction_single(g_2, e_d):
    lam = 1.0
    eta_target = 320.0
    kappa_b = 10.0
    kappa_a = 1.0

    # Guard: parameters must give a real, positive cat size
    eps_2 = 2 * float(g_2) * float(e_d) / kappa_b
    kappa_2 = 4 * float(g_2)**2 / kappa_b
    if kappa_2 == 0 or (eps_2 - kappa_a / 4) <= 0:
        return -100.0  # invalid region

    # T_Z: initialize in |+z⟩, long timescale (~100 us)
    res_z = measure_lifetime("+z", 200, g_2, e_d)
    szt = res_z.expects[1, :].real
    t_z = res_z.tsave
    z_fit = robust_exp_fit(t_z, szt)
    Tz = z_fit["popt"][1]

    # T_X: initialize in |+x⟩, short timescale (~1 us)
    res_x = measure_lifetime("+x", 1.0, g_2, e_d)
    sxt = res_x.expects[0, :].real
    t_x = res_x.tsave
    x_fit = robust_exp_fit(t_x, sxt)
    Tx = x_fit["popt"][1]

    if Tz <= 0 or Tx <= 0:
        return -100.0  # bad fit

    eta = Tz / Tx
    return float(jnp.log(Tz) + jnp.log(Tx) - lam * jnp.abs(jnp.log(eta) - jnp.log(eta_target)))


def lossfunction(g_2, e_d):
    """Accepts scalar or 1-D array inputs for g_2 and e_d."""
    g_2 = jnp.asarray(g_2)
    e_d = jnp.asarray(e_d)
    if g_2.ndim == 0:
        return _lossfunction_single(g_2, e_d)
    return jnp.array([_lossfunction_single(g_2[j], e_d[j]) for j in range(len(g_2))])

In [ ]:
BATCH_SIZE = 12
N_EPOCHS = 80

# ----------------------------------------
# CMA-ES setup
# ----------------------------------------

mean0 = jnp.array([1.0, 4.0])   # start near known good region (g2=1, eps_d=4)
sigma0 = 0.3

optimizer = SepCMA(
    mean=mean0,
    sigma=sigma0,
    bounds=jnp.array([
        [0.7, 2.0],   # g_2: must be > 0.625 to keep alpha real (g2*eps_d > 1.25 at eps_d=2)
        [2.0, 6.0],   # eps_d
    ]),
    population_size=BATCH_SIZE,
    seed=0,
)

# ----------------------------------------
# Logging
# ----------------------------------------
mean_history = []
reward_history = []
reward_std_history = []

# ----------------------------------------
# Training loop
# ----------------------------------------
for epoch in range(N_EPOCHS):
    # Sample population
    xs = jnp.array([optimizer.ask() for _ in range(optimizer.population_size)])

    # Evaluate each sample individually (g_2 and eps_d must be scalars for dynamiqs)
    rewards = lossfunction(xs[:, 0], xs[:, 1])

    # Tell optimizer
    solutions = [(xs[j], rewards[j]) for j in range(len(xs))]
    optimizer.tell(solutions)

    # Log
    mean_history.append(optimizer.mean.copy())
    reward_history.append(jnp.mean(rewards))
    reward_std_history.append(jnp.std(rewards))

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | mean={optimizer.mean} | avg reward={jnp.mean(rewards):.4f}")

In [ ]:
mean_history = jnp.array(mean_history)
reward_history = jnp.array(reward_history)
reward_std_history = jnp.array(reward_std_history)


# ----------------------------------------
# Plot 1: Reward vs epoch
# ----------------------------------------
epochs = jnp.arange(N_EPOCHS)

plt.figure(figsize=(7, 4))
plt.plot(epochs, reward_history, label="Mean reward")
plt.fill_between(
    epochs,
    reward_history - reward_std_history,
    reward_history + reward_std_history,
    alpha=0.2,
)
plt.xlabel("Epoch")
plt.ylabel("Reward (log infidelity)")
plt.title("CMA-ES: Reward vs Epoch")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


# ----------------------------------------
# Plot 2: Parameters vs epoch
# ----------------------------------------
plt.figure(figsize=(7, 4))

plt.plot(epochs, mean_history[:, 0], label="amp factor Δ")
plt.plot(epochs, mean_history[:, 1], label="freq. Δ (MHz)")

# true optimum
plt.axhline(0.0, linestyle="--", label="amp factor Δ optimal: 0.0")
plt.axhline(0.0, linestyle="--", label="freq Δ (MHz) optimal: 0.0")

plt.xlabel("Epoch")
plt.ylabel("Parameter value")
plt.title("CMA-ES Parameter Convergence")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()